# Laboratorio EEG - Bloque 2: guía para análisis y Machine Learning

## Objetivo

Este notebook es un **código de referencia** para completar el pipeline del laboratorio a partir de todas las señales EEG adquiridas.

La intención no es que el grupo dedique la mayor parte del tiempo a programar. El código deja listas las herramientas principales para que puedan concentrarse en:

- Revisar la calidad del dataset
- Comparar las tres condiciones
- Interpretar las características extraídas
- Comparar modelos de clasificación
- Discutir las métricas y las limitaciones del experimento

El pipeline utilizado será:

**archivos EEG -> control de calidad -> preprocesamiento -> ventanas -> características -> validación por sujeto -> modelos -> métricas**

---

## Principios importantes

1. Cada archivo EEG corresponde a **una sola clase**:
   - clase 0: escucha pasiva;
   - clase 1: 0-back;
   - clase 2: 2-back.

2. Los dos pulsos de `I1` delimitan el intervalo útil de la actividad.

3. Las ventanas con overlap provenientes del mismo sujeto **no deben repartirse aleatoriamente entre entrenamiento y prueba**.

4. Las métricas conductuales del N-back, como Balanced Accuracy, tiempo de reacción y esfuerzo mental, se utilizarán como **control de calidad e información complementaria**, no como variables de entrada del clasificador EEG.

## 1. Librerías

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import spearmanr

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

warnings.filterwarnings("ignore")

## 2. Configuración general

Antes de ejecutar el pipeline, revisen esta celda.

La convención esperada para los archivos EEG es:

`grupo3_clase2_trial1.txt`

Si cada grupo utilizó siempre al mismo participante, `group_id` puede utilizarse como `subject_id`. Si no fue así, completen `SUBJECT_MAP`.

In [ ]:
# RUTAS

DATA_DIR = Path("dataset_eeg")

# Opcional:
# CSV que vincula cada archivo EEG con el summary.csv de su N-back.
BEHAVIOR_MANIFEST_FILE = None

OUTPUT_DIR = Path("resultados_bloque2")
SAVE_OUTPUTS = True

# ARCHIVOS Y CLASES

FILE_PATTERN = re.compile(
    r"grupo(?P<group>\d+)_clase(?P<label>[012])_trial(?P<trial>\d+)",
    flags=re.IGNORECASE
)

LABEL_MAP = {
    0: "Escucha pasiva",
    1: "0-back",
    2: "2-back",
}

CLASS_ORDER = [0, 1, 2]

EEG_COL = "A4"
BUTTON_COL = "I1"

# SUJETOS

# Queda vacío porque cada grupo corresponde a un único participante.
SUBJECT_MAP = {}


# MARCADORES

BUTTON_THRESHOLD = 0.5
MIN_EVENT_DISTANCE_S = 0.5
MARGIN_S = 0.5

# Si un archivo tiene marcadores incorrectos, se puede indicar manualmente
# el intervalo útil en segundos:
# MANUAL_INTERVALS_S = {
#     "grupo3_clase2_trial1.txt": (5.2, 50.3),
# }
MANUAL_INTERVALS_S = {}


# PREPROCESAMIENTO

LOWCUT_HZ = 2
HIGHCUT_HZ = 30
FILTER_ORDER = 4


# VENTANAS

WINDOW_S = 4
OVERLAP = 0.50


# CONTROL DE CALIDAD CONDUCTUAL

BEHAVIOR_BA_THRESHOLD = 0.75

# Si se proporciona BEHAVIOR_MANIFEST_FILE, los ensayos activos
# con BA menor al umbral quedarán marcados como no válidos.
USE_BEHAVIOR_QC = True


# MACHINE LEARNING

RANDOM_STATE = 42 # Seed

# Validación dejando un sujeto completo para prueba.
CV_GROUP_COL = "subject_id"

## 3. Conversión ADC a µV

Copien aquí la conversión que completaron y verificaron en el Bloque 1 utilizando el manual del módulo EEG.

Esta es una de las pocas partes que se conserva deliberadamente del trabajo realizado por ustedes.

In [ ]:
# OBTENER DEL BLOQUE 1

ADC_BITS = None
VCC = None
EEG_GAIN = None

def adc_to_uv(adc_values):
    '''
    Convertir las cuentas digitales del canal EEG a microvoltios.

    Reemplacen esta función por la conversión correcta desarrollada
    en el Bloque 1.
    '''
    raise NotImplementedError(
        "Copie aquí su conversión ADC -> µV del Bloque 1."
    )

## 4. Funciones auxiliares

Estas funciones implementan la lectura de OpenSignals, detección de marcadores, filtrado, ventaneo y extracción de características.

No es necesario modificar esta sección salvo que quieran probar otras alternativas.

In [ ]:
def read_opensignals_txt(filepath):
    '''
    Lee un .txt exportado desde OpenSignals.
    '''
    filepath = Path(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    metadata = {}

    for line in lines:
        if line.startswith("# {"):
            try:
                metadata = json.loads(line[2:].strip())
            except json.JSONDecodeError:
                pass
            break

    df = pd.read_csv(
        filepath,
        sep="\t",
        comment="#",
        header=None,
        engine="python"
    )

    if df.shape[1] > 0 and df.iloc[:, -1].isna().all():
        df = df.iloc[:, :-1]

    columns = None

    if metadata:
        try:
            device_key = list(metadata.keys())[0]
            columns = metadata[device_key].get("column", None)
        except Exception:
            columns = None

    if columns is not None and len(columns) == df.shape[1]:
        df.columns = columns
    else:
        df.columns = [f"col_{i}" for i in range(df.shape[1])]

    return metadata, df


def get_sampling_rate(metadata):
    if not metadata:
        return None

    device_key = list(metadata.keys())[0]
    value = metadata[device_key].get("sampling rate", None)

    if value is None:
        return None

    return float(value)


def parse_eeg_filename(filepath):
    name = Path(filepath).stem
    match = FILE_PATTERN.search(name)

    if match is None:
        return None

    info = match.groupdict()

    group_id = str(info["group"])
    label = int(info["label"])
    trial_id = int(info["trial"])

    subject_id = SUBJECT_MAP.get(group_id, group_id)

    return {
        "group_id": group_id,
        "subject_id": str(subject_id),
        "label": label,
        "phase_name": LABEL_MAP[label],
        "trial_id": trial_id,
    }


def detect_rising_edges(
    button_signal,
    fs,
    threshold=0.5,
    min_event_distance_s=0.5
):
    button_signal = np.asarray(button_signal)

    binary = (button_signal > threshold).astype(int)

    rising_edges = np.where(
        np.diff(binary, prepend=binary[0]) == 1
    )[0]

    min_samples = int(min_event_distance_s * fs)

    filtered = []

    for idx in rising_edges:
        if len(filtered) == 0 or idx - filtered[-1] >= min_samples:
            filtered.append(idx)

    return np.asarray(filtered, dtype=int)


def bandpass_filter(x, fs, lowcut=2, highcut=30, order=4):
    sos = signal.butter(
        order,
        [lowcut, highcut],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return signal.sosfiltfilt(sos, x)


def create_windows(x, fs, window_s=4, overlap=0.50):
    x = np.asarray(x)

    window_samples = int(window_s * fs)

    if not 0 <= overlap < 1:
        raise ValueError("overlap debe estar entre 0 y 1.")

    step_samples = int(window_samples * (1 - overlap))

    if step_samples < 1:
        raise ValueError("El paso entre ventanas debe ser >= 1 muestra.")

    windows = []
    starts = []

    for start in range(
        0,
        len(x) - window_samples + 1,
        step_samples
    ):
        end = start + window_samples
        windows.append(x[start:end])
        starts.append(start)

    return np.asarray(windows), np.asarray(starts)


def bandpower(freqs, psd, fmin, fmax):
    mask = (freqs >= fmin) & (freqs < fmax)

    if np.sum(mask) < 2:
        return np.nan

    return np.trapz(
        psd[mask],
        freqs[mask]
    )


def hjorth_parameters(x):
    x = np.asarray(x)

    dx = np.diff(x)
    ddx = np.diff(dx)

    activity = np.var(x)

    if activity <= 0 or len(dx) == 0:
        return np.nan, np.nan, np.nan

    mobility = np.sqrt(
        np.var(dx) / activity
    )

    if np.var(dx) <= 0 or len(ddx) == 0:
        complexity = np.nan
    else:
        mobility_dx = np.sqrt(
            np.var(ddx) / np.var(dx)
        )
        complexity = mobility_dx / mobility if mobility > 0 else np.nan

    return activity, mobility, complexity


def spectral_entropy(psd):
    psd = np.asarray(psd, dtype=float)

    total = np.sum(psd)

    if total <= 0:
        return np.nan

    p = psd / total
    p = p[p > 0]

    if len(p) <= 1:
        return 0.0

    return -np.sum(p * np.log2(p)) / np.log2(len(p))


def extract_features(x, fs):
    '''
    Extrae características temporales y espectrales de una ventana.
    '''
    x = np.asarray(x, dtype=float)

    activity, mobility, complexity = hjorth_parameters(x)

    # Características temporales
    features = {
        "mean": np.mean(x),
        "std": np.std(x, ddof=1),
        "rms": np.sqrt(np.mean(x ** 2)),
        "mav": np.mean(np.abs(x)),
        "waveform_length": np.sum(np.abs(np.diff(x))),
        "hjorth_activity": activity,
        "hjorth_mobility": mobility,
        "hjorth_complexity": complexity,
    }

    # Variables útiles para control de calidad.
    features["qc_max_abs_uV"] = np.max(np.abs(x))
    features["qc_ptp_uV"] = np.ptp(x)

    # PSD
    nperseg = min(
        int(2 * fs),
        len(x)
    )

    freqs, psd = signal.welch(
        x,
        fs=fs,
        nperseg=nperseg
    )

    bp_theta = bandpower(freqs, psd, 4, 8)
    bp_alpha = bandpower(freqs, psd, 8, 13)
    bp_beta = bandpower(freqs, psd, 13, 30)
    bp_total = bandpower(freqs, psd, 4, 30)

    eps = 1e-12

    features.update({
        "bp_theta": bp_theta,
        "bp_alpha": bp_alpha,
        "bp_beta": bp_beta,
        "bp_total_4_30": bp_total,
        "rp_theta": bp_theta / (bp_total + eps),
        "rp_alpha": bp_alpha / (bp_total + eps),
        "rp_beta": bp_beta / (bp_total + eps),
        "theta_alpha_ratio": bp_theta / (bp_alpha + eps),
        "theta_beta_ratio": bp_theta / (bp_beta + eps),
        "theta_alpha_beta_ratio": bp_theta / (bp_alpha + bp_beta + eps),
        "spectral_entropy": spectral_entropy(
            psd[(freqs >= 4) & (freqs <= 30)]
        ),
    })

    return features

## 5. Construcción automática del dataset EEG

El siguiente bloque procesa todos los `.txt` encontrados en `DATA_DIR`.

Para cada archivo:

- identifica grupo, clase y trial a partir del nombre
- lee la frecuencia de muestreo
- convierte a µV
- detecta los marcadores
- recorta el intervalo útil
- elimina tendencia
- filtra entre 2 y 30 Hz
- divide en ventanas
- extrae las características

Si un archivo no cumple las condiciones mínimas, se registra en la tabla de ensayos pero no se utiliza automáticamente para ML.

In [ ]:
def process_eeg_file(filepath):
    filepath = Path(filepath)

    parsed = parse_eeg_filename(filepath)

    base_row = {
        "source_file": filepath.name,
        "source_path": str(filepath),
        "status": "pending",
    }

    if parsed is None:
        base_row["status"] = "filename_error"
        return base_row, []

    base_row.update(parsed)

    try:
        metadata, df = read_opensignals_txt(filepath)

        fs = get_sampling_rate(metadata)

        if fs is None:
            base_row["status"] = "missing_fs"
            return base_row, []

        base_row["fs"] = fs
        base_row["recording_duration_s"] = len(df) / fs

        if EEG_COL not in df.columns:
            base_row["status"] = "missing_eeg_channel"
            return base_row, []

        if BUTTON_COL not in df.columns:
            base_row["status"] = "missing_button_channel"
            return base_row, []

        # ADC -> µV
        eeg_uV = adc_to_uv(
            df[EEG_COL].to_numpy(dtype=float)
        )

        # ----------------------------------------------------
        # Selección del intervalo
        # ----------------------------------------------------

        if filepath.name in MANUAL_INTERVALS_S:
            start_s, end_s = MANUAL_INTERVALS_S[filepath.name]

            start_idx = int(start_s * fs)
            end_idx = int(end_s * fs)

            marker_count = np.nan
            marker_source = "manual"

        else:
            edges = detect_rising_edges(
                df[BUTTON_COL].to_numpy(),
                fs=fs,
                threshold=BUTTON_THRESHOLD,
                min_event_distance_s=MIN_EVENT_DISTANCE_S,
            )

            marker_count = len(edges)
            base_row["marker_count"] = marker_count

            if marker_count != 2:
                base_row["status"] = "marker_error"
                return base_row, []

            start_idx = int(edges[0])
            end_idx = int(edges[1])
            marker_source = "I1"

        base_row["marker_count"] = marker_count
        base_row["marker_source"] = marker_source
        base_row["start_s"] = start_idx / fs
        base_row["end_s"] = end_idx / fs

        margin_samples = int(MARGIN_S * fs)

        start_clean = start_idx + margin_samples
        end_clean = end_idx - margin_samples

        if end_clean <= start_clean:
            base_row["status"] = "invalid_interval"
            return base_row, []

        eeg_task = eeg_uV[start_clean:end_clean]

        base_row["task_duration_s"] = len(eeg_task) / fs

        # ----------------------------------------------------
        # Preprocesamiento
        # ----------------------------------------------------

        eeg_detrended = signal.detrend(eeg_task)

        eeg_filtered = bandpass_filter(
            eeg_detrended,
            fs=fs,
            lowcut=LOWCUT_HZ,
            highcut=HIGHCUT_HZ,
            order=FILTER_ORDER,
        )

        # ----------------------------------------------------
        # Ventanas
        # ----------------------------------------------------

        windows, starts = create_windows(
            eeg_filtered,
            fs=fs,
            window_s=WINDOW_S,
            overlap=OVERLAP,
        )

        if len(windows) == 0:
            base_row["status"] = "too_short"
            return base_row, []

        base_row["n_windows"] = len(windows)
        base_row["status"] = "ok"

        feature_rows = []

        for window_id, (win, start) in enumerate(
            zip(windows, starts)
        ):
            feats = extract_features(win, fs)

            row = {
                "source_file": filepath.name,
                "recording_id": filepath.stem,
                "group_id": parsed["group_id"],
                "subject_id": parsed["subject_id"],
                "trial_id": parsed["trial_id"],
                "label": parsed["label"],
                "phase_name": parsed["phase_name"],
                "window_id": window_id,
                "window_start_s": start / fs,
                "window_end_s": (start + len(win)) / fs,
            }

            row.update(feats)
            feature_rows.append(row)

        return base_row, feature_rows

    except Exception as exc:
        base_row["status"] = f"error: {type(exc).__name__}"
        base_row["error_message"] = str(exc)
        return base_row, []


eeg_files = sorted(DATA_DIR.glob("*.txt"))

print(f"Archivos EEG encontrados: {len(eeg_files)}")

trial_rows = []
feature_rows = []

for filepath in eeg_files:
    trial_row, features_i = process_eeg_file(filepath)
    trial_rows.append(trial_row)
    feature_rows.extend(features_i)

trial_table = pd.DataFrame(trial_rows)
features_df = pd.DataFrame(feature_rows)

print("\nEstado de los archivos:")
if len(trial_table) > 0:
    display(trial_table["status"].value_counts(dropna=False))

print("\nTabla de ensayos:")
display(trial_table)

print("\nDataset de ventanas:")
print(features_df.shape)
display(features_df.head())

## 6. Integración opcional de las métricas del N-back

El script del N-back genera un `summary.csv` por prueba.

Como el nombre del archivo EEG y el nombre del `summary.csv` no necesariamente contienen el mismo identificador, la forma más segura de unirlos es mediante un archivo pequeño de correspondencias.

### Formato de `behavior_manifest.csv`

| source_file | summary_file |
|---|---|
| grupo3_clase1_trial1.txt | 20260906_G3_0back_summary.csv |
| grupo3_clase2_trial1.txt | 20260906_G3_2back_summary.csv |

Las métricas conductuales no se usarán para entrenar el modelo EEG.

In [ ]:
def load_behavior_manifest(manifest_file):
    manifest_file = Path(manifest_file)

    mapping = pd.read_csv(manifest_file)

    required = {"source_file", "summary_file"}

    if not required.issubset(mapping.columns):
        raise ValueError(
            "behavior_manifest.csv debe contener "
            "'source_file' y 'summary_file'."
        )

    behavior_rows = []

    for _, row in mapping.iterrows():
        summary_path = Path(row["summary_file"])

        if not summary_path.is_absolute():
            summary_path = manifest_file.parent / summary_path

        summary = pd.read_csv(summary_path)

        if len(summary) == 0:
            continue

        record = summary.iloc[0].to_dict()
        record["source_file"] = row["source_file"]
        record["summary_file"] = str(summary_path)

        behavior_rows.append(record)

    return pd.DataFrame(behavior_rows)


behavior_df = pd.DataFrame()

if BEHAVIOR_MANIFEST_FILE is not None:
    behavior_df = load_behavior_manifest(
        BEHAVIOR_MANIFEST_FILE
    )

    print("Métricas conductuales cargadas:")
    display(behavior_df.head())

    behavior_cols = [
        "source_file",
        "balanced_accuracy",
        "mean_hit_rt_ms",
        "median_hit_rt_ms",
        "mental_effort_1_to_7",
        "mode",
    ]

    available_behavior_cols = [
        c for c in behavior_cols
        if c in behavior_df.columns
    ]

    trial_table = trial_table.merge(
        behavior_df[available_behavior_cols],
        on="source_file",
        how="left",
    )

else:
    print(
        "No se proporcionó behavior_manifest.csv. "
        "El pipeline EEG puede continuar sin esta sección."
    )

### Control de calidad conductual

Para 0-back y 2-back, una Balanced Accuracy baja indica que el participante no ejecutó adecuadamente la tarea.

El código siguiente marca los ensayos activos con `BA >= 0.75` como válidos.

La escucha pasiva no tiene Balanced Accuracy y no se descarta por este criterio.

In [ ]:
trial_table["eeg_valid"] = (
    trial_table["status"] == "ok"
)

trial_table["behavior_valid"] = True

if (
    USE_BEHAVIOR_QC
    and "balanced_accuracy" in trial_table.columns
):
    active_mask = trial_table["label"].isin([1, 2])

    trial_table.loc[
        active_mask,
        "behavior_valid"
    ] = (
        trial_table.loc[
            active_mask,
            "balanced_accuracy"
        ] >= BEHAVIOR_BA_THRESHOLD
    )

trial_table["trial_valid"] = (
    trial_table["eeg_valid"]
    & trial_table["behavior_valid"]
)

display(
    trial_table[
        [
            c for c in [
                "source_file",
                "subject_id",
                "label",
                "phase_name",
                "task_duration_s",
                "balanced_accuracy",
                "median_hit_rt_ms",
                "mental_effort_1_to_7",
                "eeg_valid",
                "behavior_valid",
                "trial_valid",
            ]
            if c in trial_table.columns
        ]
    ]
)

## 7. Análisis exploratorio y control de calidad

Antes de entrenar un modelo, revisen primero qué datos tienen realmente.

Algunas preguntas útiles para la primera parte del reporte:

- ¿Todos los sujetos tienen las tres clases?
- ¿Hay aproximadamente el mismo número de trials por clase?
- ¿Cuántos archivos fueron descartados?
- ¿La duración útil de los ensayos es similar?
- ¿El desempeño conductual fue menor en 2-back que en 0-back?
- ¿El esfuerzo mental reportado aumenta con la dificultad?

In [ ]:
# Conteo de trials válidos por clase.

valid_trials = trial_table[
    trial_table["trial_valid"]
].copy()

trial_counts = (
    valid_trials
    .groupby(["label", "phase_name"])
    .size()
    .reset_index(name="n_trials")
)

display(trial_counts)

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    trial_counts["phase_name"],
    trial_counts["n_trials"]
)

ax.set_title("Número de trials válidos por clase")
ax.set_ylabel("Trials")
ax.set_xlabel("Clase")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Verificar que cada sujeto tenga datos de las tres clases.

subject_class_table = pd.crosstab(
    valid_trials["subject_id"],
    valid_trials["phase_name"]
)

print("Trials válidos por sujeto y clase:")
display(subject_class_table)

In [ ]:
# Duración útil por clase.

if len(valid_trials) > 0:
    duration_summary = (
        valid_trials
        .groupby("phase_name")["task_duration_s"]
        .agg(["count", "mean", "std", "min", "max"])
    )

    display(duration_summary)

### 7.1 Análisis conductual opcional

Esta parte es especialmente útil para comprobar que las condiciones realmente generaron diferentes niveles de dificultad.

No interpreten estas métricas como características EEG.

In [ ]:
behavior_columns = {
    "balanced_accuracy",
    "median_hit_rt_ms",
    "mental_effort_1_to_7",
}

if behavior_columns.intersection(trial_table.columns):

    cols = [
        c for c in [
            "source_file",
            "subject_id",
            "phase_name",
            "balanced_accuracy",
            "median_hit_rt_ms",
            "mental_effort_1_to_7",
        ]
        if c in trial_table.columns
    ]

    display(
        trial_table[cols]
        .sort_values(["subject_id", "phase_name"])
    )

    for metric in [
        "balanced_accuracy",
        "median_hit_rt_ms",
        "mental_effort_1_to_7",
    ]:
        if metric not in trial_table.columns:
            continue

        plot_df = trial_table.dropna(
            subset=[metric, "phase_name"]
        )

        if len(plot_df) == 0:
            continue

        classes = [
            LABEL_MAP[k]
            for k in CLASS_ORDER
            if LABEL_MAP[k] in plot_df["phase_name"].unique()
        ]

        data = [
            plot_df.loc[
                plot_df["phase_name"] == cls,
                metric
            ].to_numpy()
            for cls in classes
        ]

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.boxplot(
            data,
            tick_labels=classes,
            showmeans=True
        )

        ax.set_title(f"{metric} por condición")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=20)

        plt.tight_layout()
        plt.show()

## 8. Selección del dataset de características

Solo se conservarán ventanas pertenecientes a trials válidos.

Los campos `qc_max_abs_uV` y `qc_ptp_uV` se mantienen para inspección de artefactos, pero no se usarán inicialmente como características del modelo.

In [ ]:
valid_files = set(
    trial_table.loc[
        trial_table["trial_valid"],
        "source_file"
    ]
)

features_model = features_df[
    features_df["source_file"].isin(valid_files)
].copy()

print("Ventanas válidas:", len(features_model))

print("\nVentanas por clase:")
display(
    features_model["phase_name"]
    .value_counts()
    .rename("n_windows")
)

print("\nVentanas por sujeto y clase:")
display(
    pd.crosstab(
        features_model["subject_id"],
        features_model["phase_name"]
    )
)

### 8.1 Inspección sencilla de amplitudes

Debido al montaje frontal, parpadeos, movimientos o actividad muscular pueden aparecer con amplitudes elevadas.

No se define un umbral universal de descarte en este notebook. Revisen la distribución de estas variables y documenten si decidieron retirar algún trial o ventana.

In [ ]:
qc_cols = [
    "qc_max_abs_uV",
    "qc_ptp_uV",
]

display(
    features_model[qc_cols]
    .describe()
    .T
)

features_model[qc_cols].hist(
    figsize=(10, 4),
    bins=30
)

plt.suptitle("Indicadores simples para revisión de artefactos")
plt.tight_layout()
plt.show()

## 9. Exploración de características EEG

Para interpretar mejor los datos, es preferible comparar primero las características antes de ejecutar los clasificadores.

Las gráficas siguientes utilizan el **promedio de cada trial**, no cada ventana como si fuera una observación independiente.

In [ ]:
MODEL_FEATURES = [
    # Tiempo
    "std",
    "rms",
    "mav",
    "waveform_length",
    "hjorth_activity",
    "hjorth_mobility",
    "hjorth_complexity",

    # Frecuencia
    "bp_theta",
    "bp_alpha",
    "bp_beta",
    "rp_theta",
    "rp_alpha",
    "rp_beta",
    "theta_alpha_ratio",
    "theta_beta_ratio",
    "theta_alpha_beta_ratio",
    "spectral_entropy",
]

MODEL_FEATURES = [
    c for c in MODEL_FEATURES
    if c in features_model.columns
]

print(f"Número de características seleccionadas: {len(MODEL_FEATURES)}")
print(MODEL_FEATURES)

In [ ]:
trial_feature_means = (
    features_model
    .groupby(
        [
            "source_file",
            "recording_id",
            "subject_id",
            "label",
            "phase_name",
        ],
        as_index=False
    )[MODEL_FEATURES]
    .mean()
)

display(trial_feature_means.head())

In [ ]:
# Cambien esta lista para explorar otras características.

FEATURES_TO_PLOT = [
    "rp_theta",
    "rp_alpha",
    "rp_beta",
    "theta_alpha_beta_ratio",
]

features_to_plot = [
    c for c in FEATURES_TO_PLOT
    if c in trial_feature_means.columns
]

for feature in features_to_plot:

    classes = [
        LABEL_MAP[k]
        for k in CLASS_ORDER
        if LABEL_MAP[k]
        in trial_feature_means["phase_name"].unique()
    ]

    data = [
        trial_feature_means.loc[
            trial_feature_means["phase_name"] == cls,
            feature
        ].dropna().to_numpy()
        for cls in classes
    ]

    fig, ax = plt.subplots(figsize=(8, 4))

    ax.boxplot(
        data,
        tick_labels=classes,
        showmeans=True
    )

    ax.set_title(f"{feature} por condición")
    ax.set_ylabel(feature)
    ax.tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

### Interpretación sugerida

Antes del ML, identifiquen (útil para la parte del reporte de extracción de características):

- qué características presentan diferencias visibles entre las clases
- qué características tienen gran variabilidad entre sujetos
- si las tendencias observadas son consistentes entre trials
- si una diferencia podría estar relacionada con artefactos y no necesariamente con actividad cortical

## 10. Relación opcional entre EEG y desempeño conductual

Esta sección agrega las métricas conductuales a nivel de trial únicamente para análisis.

Ejemplos de preguntas:

- ¿Mayor esfuerzo mental se relaciona con mayor potencia theta relativa?
- ¿Los trials con mayor tiempo de reacción presentan algún cambio espectral?
- ¿El desempeño conductual confirma que 2-back fue más difícil que 0-back?

No es obligatorio encontrar correlaciones significativas.

In [ ]:
behavior_metrics_available = [
    c for c in [
        "balanced_accuracy",
        "median_hit_rt_ms",
        "mental_effort_1_to_7",
    ]
    if c in trial_table.columns
]

if behavior_metrics_available:

    analysis_trial = trial_feature_means.merge(
        trial_table[
            [
                "source_file",
                *behavior_metrics_available
            ]
        ],
        on="source_file",
        how="left"
    )

    display(
        analysis_trial.head()
    )

    EEG_VARIABLE_FOR_CORRELATION = "rp_theta"

    if EEG_VARIABLE_FOR_CORRELATION in analysis_trial.columns:

        for behavior_metric in behavior_metrics_available:

            tmp = analysis_trial[
                [
                    EEG_VARIABLE_FOR_CORRELATION,
                    behavior_metric
                ]
            ].dropna()

            if len(tmp) >= 4:
                rho, p = spearmanr(
                    tmp[EEG_VARIABLE_FOR_CORRELATION],
                    tmp[behavior_metric]
                )

                print(
                    f"{EEG_VARIABLE_FOR_CORRELATION} vs "
                    f"{behavior_metric}: "
                    f"rho={rho:.3f}, p={p:.4f}"
                )
else:
    print(
        "No se cargaron métricas conductuales. "
        "Esta sección se omite."
    )

## 11. Preparación del Machine Learning

La validación recomendada es **Leave-One-Subject-Out**:

- todas las ventanas de un sujeto quedan juntas
- el modelo se entrena con los demás sujetos
- el sujeto dejado fuera se utiliza únicamente como prueba

Esto evita que ventanas solapadas o muy similares del mismo participante aparezcan simultáneamente en train y test.

In [ ]:
if features_model["subject_id"].nunique() < 2:
    raise ValueError(
        "Se necesitan al menos dos subject_id para Leave-One-Subject-Out. "
        "Revise SUBJECT_MAP o integre los datos de más participantes."
    )

X = features_model[MODEL_FEATURES].copy()
y = features_model["label"].astype(int).copy()
groups = features_model[CV_GROUP_COL].astype(str).copy()

print("X:", X.shape)
print("Sujetos:", groups.nunique())
print("Clases:", sorted(y.unique()))

## 12. Modelos de referencia

Se incluyen:

- Dummy: baseline de referencia (Investigar sobre como se usa un Dummy classifier)
- Logistic Regression: modelo lineal sencillo
- SVM con kernel RBF: modelo no lineal
- Random Forest: modelo basado en árboles

Pueden reportar todos o seleccionar al menos dos modelos reales para discutirlos.

In [ ]:
models = {
    "Dummy": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", DummyClassifier(
            strategy="prior",
            random_state=RANDOM_STATE
        ))
    ]),

    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "SVM_RBF": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight="balanced"
        ))
    ]),

    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
}

## 13. Validación Leave-One-Subject-Out

Se calcularán métricas por fold.

Para este experimento se recomienda prestar especial atención a:

- Balanced Accuracy
- F1 macro
- matriz de confusión
- variabilidad de los resultados entre sujetos

El promedio entre folds representa mejor la generalización entre sujetos que una única métrica calculada juntando todas las ventanas.

In [ ]:
logo = LeaveOneGroupOut()

fold_rows = []
prediction_rows = []

for model_name, model_template in models.items():

    for fold_idx, (train_idx, test_idx) in enumerate(
        logo.split(X, y, groups=groups),
        start=1
    ):
        model = clone(model_template)

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        test_subject = groups.iloc[test_idx].iloc[0]

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        fold_rows.append({
            "model": model_name,
            "fold": fold_idx,
            "test_subject": test_subject,
            "n_train_windows": len(train_idx),
            "n_test_windows": len(test_idx),

            "accuracy": accuracy_score(
                y_test,
                y_pred
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_test,
                y_pred
            ),

            "precision_macro": precision_score(
                y_test,
                y_pred,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0
            ),

            "recall_macro": recall_score(
                y_test,
                y_pred,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0
            ),

            "f1_macro": f1_score(
                y_test,
                y_pred,
                labels=CLASS_ORDER,
                average="macro",
                zero_division=0
            ),
        })

        test_meta = features_model.iloc[
            test_idx
        ][
            [
                "source_file",
                "recording_id",
                "subject_id",
                "window_id",
            ]
        ].reset_index(drop=True)

        for i in range(len(test_idx)):
            prediction_rows.append({
                "model": model_name,
                "fold": fold_idx,
                "test_subject": test_subject,
                "source_file": test_meta.loc[i, "source_file"],
                "recording_id": test_meta.loc[i, "recording_id"],
                "window_id": test_meta.loc[i, "window_id"],
                "y_true": int(y_test.iloc[i]),
                "y_pred": int(y_pred[i]),
            })


fold_metrics_df = pd.DataFrame(
    fold_rows
)

predictions_df = pd.DataFrame(
    prediction_rows
)

display(fold_metrics_df.head())

## 14. Resumen de métricas

Se reportará el promedio y desviación estándar entre sujetos.

In [ ]:
metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
]

summary_rows = []

for model_name, df_model in fold_metrics_df.groupby("model"):

    row = {
        "model": model_name,
        "n_folds": len(df_model),
    }

    for metric in metric_cols:
        row[f"{metric}_mean"] = df_model[metric].mean()
        row[f"{metric}_std"] = df_model[metric].std()

    summary_rows.append(row)

cv_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        "balanced_accuracy_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(cv_summary_df)

In [ ]:
# Balanced Accuracy por sujeto.

for model_name, df_model in fold_metrics_df.groupby("model"):

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.bar(
        df_model["test_subject"].astype(str),
        df_model["balanced_accuracy"]
    )

    ax.axhline(
        1 / len(CLASS_ORDER),
        linestyle="--",
        label="Referencia 3 clases"
    )

    ax.set_ylim(0, 1)
    ax.set_title(
        f"Balanced Accuracy por sujeto - {model_name}"
    )
    ax.set_xlabel("Sujeto dejado para prueba")
    ax.set_ylabel("Balanced Accuracy")
    ax.legend()

    plt.tight_layout()
    plt.show()

## 15. Matrices de confusión

La matriz de confusión permite identificar qué clases se confunden con mayor frecuencia.

**¿La confusión ocurre principalmente entre escucha pasiva y 0-back, o entre 0-back y 2-back?**

In [ ]:
for model_name in models.keys():

    pred_model = predictions_df[
        predictions_df["model"] == model_name
    ]

    cm = confusion_matrix(
        pred_model["y_true"],
        pred_model["y_pred"],
        labels=CLASS_ORDER
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            LABEL_MAP[k]
            for k in CLASS_ORDER
        ]
    )

    fig, ax = plt.subplots(figsize=(7, 6))

    disp.plot(
        ax=ax,
        colorbar=False,
        xticks_rotation=25
    )

    ax.set_title(
        f"Matriz de confusión - {model_name}"
    )

    plt.tight_layout()
    plt.show()

## 16. Reporte detallado del mejor modelo

El criterio utilizado aquí es la **Balanced Accuracy promedio entre folds**.

Pueden elegir otro criterio si lo justifican.

In [ ]:
best_model_name = cv_summary_df.iloc[0]["model"]

print(
    "Mejor modelo según Balanced Accuracy promedio:",
    best_model_name
)

best_predictions = predictions_df[
    predictions_df["model"] == best_model_name
]

print(
    classification_report(
        best_predictions["y_true"],
        best_predictions["y_pred"],
        labels=CLASS_ORDER,
        target_names=[
            LABEL_MAP[k]
            for k in CLASS_ORDER
        ],
        zero_division=0
    )
)

## 17. Guardado de resultados

Los CSV generados permiten preparar tablas y figuras para el reporte sin volver a entrenar los modelos.

In [ ]:
if SAVE_OUTPUTS:

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    trial_table.to_csv(
        OUTPUT_DIR / "trial_table.csv",
        index=False
    )

    features_model.to_csv(
        OUTPUT_DIR / "window_features.csv",
        index=False
    )

    trial_feature_means.to_csv(
        OUTPUT_DIR / "trial_feature_means.csv",
        index=False
    )

    fold_metrics_df.to_csv(
        OUTPUT_DIR / "cv_fold_metrics.csv",
        index=False
    )

    cv_summary_df.to_csv(
        OUTPUT_DIR / "cv_summary.csv",
        index=False
    )

    predictions_df.to_csv(
        OUTPUT_DIR / "cv_predictions.csv",
        index=False
    )

    print(
        f"Resultados guardados en: {OUTPUT_DIR.resolve()}"
    )


El código anterior es solo una herramienta. El puntaje del reporte se centra en **presentar e interpretar los resultados obtenidos**.
